[Reference](https://medium.com/@Rohan_Dutt/10-data-compression-techniques-for-large-scale-data-systems-that-reduce-storage-costs-0888ea207a09$0)

# 1. Columnar Compression for Structured Data


In [1]:
import pandas as pd
import numpy as np
import os

# Generate 10M row dataset simulating web logs
n_rows = 10_000_000
df = pd.DataFrame({
    'user_id': np.random.randint(1, 1000000, n_rows),
    'action': np.random.choice(['click', 'view', 'purchase', 'logout'], n_rows),
    'timestamp': pd.date_range('2024-01-01', periods=n_rows, freq='s'),
    'value': np.random.randn(n_rows)
})

# CSV storage (row-oriented)
df.to_csv('logs.csv', index=False)
csv_size = os.path.getsize('logs.csv')

# Parquet storage (columnar + compression)
df.to_parquet('logs.parquet', compression='snappy', engine='pyarrow')
parquet_size = os.path.getsize('logs.parquet')

print(f"CSV size: {csv_size / 1e9:.2f} GB")
print(f"Parquet size: {parquet_size / 1e9:.2f} GB")
print(f"Compression ratio: {csv_size / parquet_size:.2f}x")
print(f"Storage savings: {(1 - parquet_size/csv_size) * 100:.1f}%")
# Query specific columns without reading entire file
df_action = pd.read_parquet('logs.parquet', columns=['action'])

CSV size: 0.53 GB
Parquet size: 0.20 GB
Compression ratio: 2.63x
Storage savings: 62.0%


# 2. Dictionary Encoding for Repetitive Values


In [2]:
import pandas as pd
import numpy as np

# Simulate web server logs with repetitive strings
log_data = {
    'status_code': ['200 OK'] * 500000 + ['404 Not Found'] * 300000 + ['500 Internal Server Error'] * 200000,
    'country': ['United States'] * 700000 + ['United Kingdom'] * 300000,
}
df_original = pd.DataFrame(log_data)

# Memory usage with string storage (object dtype)
original_memory = df_original.memory_usage(deep=True).sum()

# Apply dictionary encoding using categorical
df_encoded = df_original.copy()
for col in df_original.columns:
    df_encoded[col] = df_encoded[col].astype('category')

encoded_memory = df_encoded.memory_usage(deep=True).sum()

print(f"Original memory: {original_memory / 1e6:.2f} MB")
print(f"Encoded memory: {encoded_memory / 1e6:.2f} MB")
print(f"Reduction: {(1 - encoded_memory/original_memory) * 100:.1f}%")
print(f"Status code categories: {df_encoded['status_code'].nunique()} unique values")

# On-disk persistence
df_original.to_parquet('logs_strings.parquet')
df_encoded.to_parquet('logs_dict.parquet')

string_size = os.path.getsize('logs_strings.parquet')
dict_size = os.path.getsize('logs_dict.parquet')
print(f"On-disk reduction: {(1 - dict_size/string_size) * 100:.1f}%")

Original memory: 123.20 MB
Encoded memory: 2.00 MB
Reduction: 98.4%
Status code categories: 3 unique values
On-disk reduction: -12.4%


# 3. Delta Encoding for Time-Series Data


In [3]:
import numpy as np
import struct

# Generate realistic time-series timestamps (1-second intervals)
base_timestamp = 1704067200
timestamps = np.arange(base_timestamp, base_timestamp + 1_000_000, dtype=np.int64)

# Original storage: 8 bytes per timestamp
original_size = len(timestamps) * timestamps.dtype.itemsize

# Delta encode (first value + differences)
deltas = np.diff(timestamps, prepend=timestamps[0])
# Since deltas are constant (1), store as int8
delta_bytes = deltas.astype(np.int8).tobytes()

encoded_size = len(delta_bytes)

print(f"Original size: {original_size / 1e6:.2f} MB")
print(f"Delta encoded: {encoded_size / 1e6:.2f} MB")
print(f"Compression ratio: {original_size / encoded_size:.2f}x")

# Decode: cumulative sum of deltas
decoded = np.cumsum(np.frombuffer(delta_bytes, dtype=np.int8)) + timestamps[0]
assert np.array_equal(decoded, timestamps)

# Real-world IoT sensor with sparse values (mostly 0)
sensor_data = np.zeros(1_000_000, dtype=np.int32)
sensor_data[::1000] = np.random.randint(1, 100, 1000)  # 0.1% non-zero

# Delta + RLE for sparse data
diffs = np.diff(sensor_data, prepend=0)
rle_data = []
current_val = diffs[0]
count = 1
for val in diffs[1:]:
    if val == current_val:
        count += 1
    else:
        rle_data.append((current_val, count))
        current_val = val
        count = 1
rle_data.append((current_val, count))

# Estimate storage: 8 bytes per run (4 value + 4 count)
rle_size = len(rle_data) * 8
print(f"Sparse data RLE size: {rle_size / 1e6:.3f} MB vs {sensor_data.nbytes / 1e6:.2f} MB original")

Original size: 8.00 MB
Delta encoded: 1.00 MB
Compression ratio: 8.00x
Sparse data RLE size: 0.024 MB vs 4.00 MB original


# 4. Zstandard (Zstd) Over Gzip


In [4]:
import zstandard as zstd
import gzip
import time

# Simulate large log file (200 MB uncompressed)
log_data = b"INFO: Request processed successfully\n" * 5_000_000
log_data += b"ERROR: Database connection timeout\n" * 2_000_000
log_data += b"WARN: High memory usage detected\n" * 3_000_000

# Zstandard compression level 3 (fast)
zstd_start = time.time()
zstd_compressed = zstd.compress(log_data, level=3)
zstd_time = time.time() - zstd_start

# Gzip compression level 6 (default)
gzip_start = time.time()
gzip_compressed = gzip.compress(log_data, compresslevel=6)
gzip_time = time.time() - gzip_start

print(f"Original size: {len(log_data) / 1e6:.2f} MB")
print(f"Zstd (lv3): {len(zstd_compressed) / 1e6:.2f} MB in {zstd_time*1000:.1f} ms")
print(f"Gzip (lv6): {len(gzip_compressed) / 1e6:.2f} MB in {gzip_time*1000:.1f} ms")
print(f"Zstd compress speed: {gzip_time / zstd_time:.1f}x faster")
print(f"Zstd size improvement: {(1 - len(zstd_compressed)/len(gzip_compressed)) * 100:.1f}%")

# Decompression speed comparison
zstd_decomp_start = time.time()
zstd.decompress(zstd_compressed)
zstd_decomp_time = time.time() - zstd_decomp_start

gzip_decomp_start = time.time()
gzip.decompress(gzip_compressed)
gzip_decomp_time = time.time() - gzip_decomp_start

print(f"Zstd decompress speed: {gzip_decomp_time / zstd_decomp_time:.1f}x faster")

Original size: 354.00 MB
Zstd (lv3): 0.03 MB in 102.8 ms
Gzip (lv6): 1.03 MB in 2370.0 ms
Zstd compress speed: 23.0x faster
Zstd size improvement: 96.8%
Zstd decompress speed: 3.7x faster


# 5. Brotli for Text/Web Assets


In [5]:
import brotli
import gzip
import json

# Generate realistic API response with repetitive JSON structure
api_response = {
    "products": [
        {
            "id": i,
            "name": f"Product_{i}",
            "category": "Electronics",
            "currency": "USD",
            "description": "High-quality electronic device with advanced features"
        }
        for i in range(50000)
    ]
}
json_data = json.dumps(api_response).encode('utf-8')

# Brotli compression (quality 6 is common)
brotli_compressed = brotli.compress(json_data, quality=6)

# Gzip compression for comparison
gzip_compressed = gzip.compress(json_data, compresslevel=6)

original_mb = len(json_data) / 1e6
brotli_mb = len(brotli_compressed) / 1e6
gzip_mb = len(gzip_compressed) / 1e6

print(f"Original JSON: {original_mb:.2f} MB")
print(f"Brotli compressed: {brotli_mb:.2f} MB")
print(f"Gzip compressed: {gzip_mb:.2f} MB")
print(f"Brotli advantage: {(gzip_mb - brotli_mb) / gzip_mb * 100:.1f}% size reduction")

# CDN cost impact
monthly_requests = 10_000_000
egress_cost_per_gb = 0.08
brotli_savings = (gzip_mb - brotli_mb) * monthly_requests / 1000 * egress_cost_per_gb
print(f"Monthly CDN cost savings: ${brotli_savings:.2f}")

Original JSON: 7.88 MB
Brotli compressed: 0.11 MB
Gzip compressed: 0.29 MB
Brotli advantage: 61.1% size reduction
Monthly CDN cost savings: $141.28


# 6. Quantization for Machine Learning Data


In [6]:
import numpy as np
import tempfile
import os

# Simulate large neural network weights (500M parameters)
np.random.seed(42)
weights = np.random.randn(500_000_000).astype(np.float32)

# Save original
with tempfile.NamedTemporaryFile(delete=False) as f:
    weights.tofile(f)
    original_path = f.name
original_size = os.path.getsize(original_path)

# Quantize to float16
weights_f16 = weights.astype(np.float16)
with tempfile.NamedTemporaryFile(delete=False) as f:
    weights_f16.tofile(f)
    f16_path = f.name
f16_size = os.path.getsize(f16_path)

# Quantize to int8 (with scale factor for demonstration)
scale = np.max(np.abs(weights)) / 127
weights_int8 = (weights / scale).clip(-128, 127).astype(np.int8)
with tempfile.NamedTemporaryFile(delete=False) as f:
    weights_int8.tofile(f)
    int8_path = f.name
int8_size = os.path.getsize(int8_path)

print(f"float32: {original_size / 1e9:.2f} GB")
print(f"float16: {f16_size / 1e9:.2f} GB ({original_size/f16_size:.1f}x reduction)")
print(f"int8: {int8_size / 1e9:.2f} GB ({original_size/int8_size:.1f}x reduction)")

# Measure quantization error
f16_error = np.mean(np.abs(weights - weights_f16.astype(np.float32)))
int8_error = np.mean(np.abs(weights - (weights_int8.astype(np.float32) * scale)))

print(f"float16 avg error: {f16_error:.8f} (almost negligible)")
print(f"int8 avg error: {int8_error:.6f} (acceptable for inference)")

# Cleanup
for p in [original_path, f16_path, int8_path]:
    os.unlink(p)

float32: 2.00 GB
float16: 1.00 GB (2.0x reduction)
int8: 0.50 GB (4.0x reduction)
float16 avg error: 0.00014053 (almost negligible)
int8 avg error: 0.023278 (acceptable for inference)


# 7. Chunking + Deduplication


In [7]:
import hashlib
import os

def deduplicate_chunks(data_chunks):
    """Deduplicate using SHA256 hashes."""
    unique_chunks = {}
    chunk_map = []
    for chunk in data_chunks:
        hash_digest = hashlib.sha256(chunk).hexdigest()
        if hash_digest not in unique_chunks:
            unique_chunks[hash_digest] = chunk
        chunk_map.append(hash_digest)
    return unique_chunks, chunk_map

def chunk_data(data, chunk_size=8192):
    """Split data into fixed-size chunks."""
    return [data[i:i+chunk_size] for i in range(0, len(data), chunk_size)]

# Simulate daily database backups with minor changes
backup_day1 = b"database_snapshot_v1" * 50000 + b"user_data_x" * 50000
backup_day2 = b"database_snapshot_v1" * 50000 + b"user_data_y" * 50000  # Only user data changed

# Chunk both backups
chunks1 = chunk_data(backup_day1)
chunks2 = chunk_data(backup_day2)

# Deduplicate
unique, map1 = deduplicate_chunks(chunks1)
unique, map2 = deduplicate_chunks(chunks2)  # Reuses existing chunks

total_original = len(backup_day1) + len(backup_day2)
total_stored = sum(len(c) for c in unique.values())

print(f"Original backups: {total_original / 1e6:.2f} MB")
print(f"Deduplicated: {total_stored / 1e6:.2f} MB")
print(f"Savings: {(1 - total_stored/total_original) * 100:.1f}%")
print(f"Unique chunks: {len(unique)} vs total chunks: {len(map1) + len(map2)}")

# Simulate restoration
def reconstruct(chunks_map, chunk_store):
    return b''.join(chunk_store[h] for h in chunks_map)

restored = reconstruct(map2, unique)
assert restored == backup_day2

Original backups: 3.10 MB
Deduplicated: 0.14 MB
Savings: 95.5%
Unique chunks: 18 vs total chunks: 380


# 8. LZ4 for “Good Enough” Speed


In [9]:
import lz4.frame
import gzip
import time
import numpy as np

# Simulate hot database cache entries (100 MB)
cache_data = np.random.randint(0, 100, size=25_000_000, dtype=np.int32).tobytes()

# LZ4 compression (fastest mode)
lz4_start = time.time()
lz4_compressed = lz4.frame.compress(cache_data, compression_level=0)
lz4_compress_time = time.time() - lz4_start

# LZ4 decompression
lz4_decomp_start = time.time()
lz4_decompressed = lz4.frame.decompress(lz4_compressed)
lz4_decomp_time = time.time() - lz4_decomp_start

# Gzip for comparison
gzip_start = time.time()
gzip_compressed = gzip.compress(cache_data, compresslevel=1)
gzip_compress_time = time.time() - gzip_start

gzip_decomp_start = time.time()
gzip.decompress(gzip_compressed)
gzip_decomp_time = time.time() - gzip_decomp_start

print(f"LZ4 compress: {lz4_compress_time*1000:.2f} ms")
print(f"Gzip compress: {gzip_compress_time*1000:.2f} ms")
print(f"LZ4 decompress: {lz4_decomp_time*1000:.2f} ms")
print(f"Gzip decompress: {gzip_decomp_time*1000:.2f} ms")
print(f"Decompression speedup: {gzip_decomp_time / lz4_decomp_time:.1f}x")

# Cache hit latency impact
print(f"LZ4 adds only {lz4_decomp_time*1000:.2f} ms to cache retrieval")

# 9. Tiered Compression (Hot vs. Cold Data)


In [10]:
import time
import zstandard as zstd
import lz4.frame
import os

def classify_and_compress(data, access_days_ago):
    """Apply compression based on access pattern."""
    if access_days_ago <= 1:  # Hot: accessed within 24h
        start = time.time()
        compressed = lz4.frame.compress(data, compression_level=0)
        method = "LZ4"
    elif access_days_ago <= 30:  # Warm: accessed within month
        start = time.time()
        compressed = zstd.compress(data, level=6)
        method = "Zstd-Mid"
    else:  # Cold: archived
        start = time.time()
        compressed = zstd.compress(data, level=19)
        method = "Zstd-Max"

    compress_time = time.time() - start
    return compressed, method, compress_time

# Simulate data with different access patterns
hot_data = b"recent_transaction" * 1000000  # Accessed today
warm_data = b"monthly_report" * 1000000    # Accessed last week
cold_data = b"yearly_archive" * 1000000    # Accessed last year

datasets = [
    (hot_data, 0),
    (warm_data, 7),
    (cold_data, 365)
]

total_original = 0
total_compressed = 0

for data, days in datasets:
    total_original += len(data)
    compressed, method, comp_time = classify_and_compress(data, days)
    total_compressed += len(compressed)

    print(f"{method:10s} | Original: {len(data)/1e6:5.1f} MB | "
          f"Compressed: {len(compressed)/1e6:5.1f} MB | "
          f"Ratio: {len(data)/len(compressed):4.1f}x | "
          f"Time: {comp_time*1000:5.1f} ms")

print(f"\nTotal savings: {(1 - total_compressed/total_original) * 100:.1f}%")
print(f"Storage cost: ${total_compressed/1e9*0.023*12:.2f}/year vs ${total_original/1e9*0.023*12:.2f}/year")

# 10. Erasure Coding for Redundancy


In [11]:
import numpy as np
from reedsolo import RSCodec

# Erasure coding parameters: k data shards, m parity shards
k, m = 6, 3  # 6 data + 3 parity = tolerate 3 failures
codec = RSCodec(m)

# Simulate 1MB file split into k shards
original_data = b"critical_storage_data" * (1024 * 1024 // 23)
print(f"Original data size: {len(original_data)} bytes")

# Traditional 3x replication cost
replication_overhead = len(original_data) * 3
print(f"3x replication size: {replication_overhead} bytes")

# Erasure coding
# Pad to multiple of k
if len(original_data) % k != 0:
    original_data += b'\x00' * (k - len(original_data) % k)

shard_size = len(original_data) // k
data_shards = [original_data[i*shard_size:(i+1)*shard_size] for i in range(k)]

# Encode: generates m parity shards
encoded = codec.encode(b''.join(data_shards))

# Total storage: k + m shards
total_shards = len(bytes(encoded))  # encoded already contains data + parity
ec_overhead = total_shards

print(f"Erasure coding size: {ec_overhead} bytes")
print(f"Storage overhead: {(ec_overhead/len(original_data) - 1) * 100:.1f}%")
print(f"Replication overhead: {(replication_overhead/len(original_data) - 1) * 100:.0f}%")
print(f"Savings: {(1 - ec_overhead/replication_overhead) * 100:.0f}%")

# Simulate failure: lose 2 data shards and 1 parity
corrupted = encoded[:shard_size] + b'\x00' * shard_size + \
            encoded[shard_size*2:shard_size*3] + b'\x00' * shard_size + \
            encoded[shard_size*3:shard_size*4] + \
            encoded[shard_size*4:shard_size*5] + \
            b'\x00' * shard_size + \
            encoded[shard_size*6:]

# Decode and recover
recovered = codec.decode(corrupted)
assert recovered == original_data
print("Successfully recovered from 3 shard failures!")